# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [4]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [3]:
results = nutrition_qna.query(query_texts=["pregnancy"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('is_pregnancy', True)]
Question: What are some possible physical changes that a pregnant woman could experience in the middle months of gestation?
        Answer: During weeks 13-27, you may see an increase in weight and feel more hunger. Backaches might occur frequently, along with leg cramps and heartburn.

        This Q&A pair provides information about nutrition and health topics.


[('is_pregnancy', True)]
Question: What health issues can make a pregnancy more challenging?
        Answer: There are several medical conditions that can potentially increase the risk associated with Pregnancy. These include Anemia during this stage, Hypertensive Disorders related to gestation, Diabetes Mellitus coexisting with Pregnancy, Obesity during pregnancy, as well as Adolescent or Teenage pregnancies.

        This Q&A pair provides information about nutrition and health topics.




In [17]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

In [21]:
@function_tool
def qna_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up information about malnutrition and health.

    Args:
        query: Topic or question to find information about.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(doc)

    return "Q&A about the topic:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [7]:
result = calorie_lookup_tool('bananas')
result

'Nutrition Information:\nBanana (Fruits): 89.0 calories per 100g\nBanana Juice ((Fruit)Juices): 50.0 calories per 100g\nBanana Nut Bread (Pastries,Breads&Rolls): 326.0 calories per 100g'

In [8]:
print(result)

Nutrition Information:
Banana (Fruits): 89.0 calories per 100g
Banana Juice ((Fruit)Juices): 50.0 calories per 100g
Banana Nut Bread (Pastries,Breads&Rolls): 326.0 calories per 100g


In [12]:
result = qna_lookup_tool('pregnancy')
print(result)

Q&A about the topic:
Question: What are some possible physical changes that a pregnant woman could experience in the middle months of gestation?
        Answer: During weeks 13-27, you may see an increase in weight and feel more hunger. Backaches might occur frequently, along with leg cramps and heartburn.

        This Q&A pair provides information about nutrition and health topics.
Question: What health issues can make a pregnancy more challenging?
        Answer: There are several medical conditions that can potentially increase the risk associated with Pregnancy. These include Anemia during this stage, Hypertensive Disorders related to gestation, Diabetes Mellitus coexisting with Pregnancy, Obesity during pregnancy, as well as Adolescent or Teenage pregnancies.

        This Q&A pair provides information about nutrition and health topics.
Question: What are some hormonal changes that take place in a mother's body during pregnancy?
        Answer: During pregnancy, there is an incre

In [22]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If you need health related information and Q&A, specially regarding malnutrition and pregnancy use the qna_lookup_tool.
    """,
    tools=[calorie_lookup_tool, qna_lookup_tool]
)

In [7]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result.final_output)

- Calories per 100 g: Banana 89 kcal; Apple 52 kcal.
- Approximate total for one medium banana (~118 g) + one medium apple (~182 g): about 200 kcal (banana ≈ 105 kcal; apple ≈ 95 kcal).


In [23]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "How many calories per day for a pregnant woman ?",
    )
    print(result.final_output)

General guidance (individual needs vary):

- First trimester: about 150 extra calories per day.
- Second and third trimesters: about 350 extra calories per day.
- Typical total intake range for many pregnant people: roughly 2,200–2,900 kcal/day, depending on pre-pregnancy weight, activity level, and health.

Always tailor to your clinician’s advice.
